In [1]:
import sys
sys.path.append("../")

import torch
from likelihoods.npll_torch import log_like_np
from models.psf import KingPSF
from utils.psf_correction import PSFCorrection

from jax.config import config
config.update("jax_enable_x64", True)

%load_ext autoreload
%autoreload 2

In [9]:
kp = KingPSF()

pc_inst = PSFCorrection(delay_compute=True, num_f_bins=15)
pc_inst.psf_r_func = lambda r: kp.psf_fermi_r(r)
pc_inst.sample_psf_max = 10.0 * kp.spe * (kp.score + kp.stail) / 2.0
pc_inst.psf_samples = 10000
pc_inst.psf_tag = "Fermi_PSF_2GeV2"
pc_inst.make_or_load_psf_corr()

f_ary = pc_inst.f_ary
df_rho_div_f_ary = pc_inst.df_rho_div_f_ary

Loading the psf correction from: /net/rcstorenfs02/ifs/rc_labs/dvorkin_lab/smsharma/mi-attribution/notebooks/psf_dir/Fermi_PSF_2GeV2.npy


In [10]:
npix = 10

pt_sum_compressed = 10 * torch.randn((npix,))
npt_compressed = 10 * torch.randn((1, npix,))
data = torch.randint(20, (npix,))
f_ary = torch.Tensor(f_ary)
df_rho_div_f_ary = torch.Tensor(df_rho_div_f_ary)

theta = torch.Tensor([[0.1, 10, -2, -3, 10, 1]])

In [11]:
log_like_np(pt_sum_compressed, theta, npt_compressed, data, f_ary, df_rho_div_f_ary)

tensor([     nan,  -7.4856,  -7.8013,  12.3722,  -2.7888,   8.5122,   8.5065,
         -6.9496, -19.7053,  -6.4565], dtype=torch.float64)

In [12]:
from likelihoods.npll_jax import log_like_np as log_like_np_jax

In [13]:
import jax.numpy as jnp

pt_sum_compressed = jnp.array(pt_sum_compressed)
npt_compressed = jnp.array(npt_compressed)
data = jnp.array(data)
f_ary = jnp.array(f_ary)
df_rho_div_f_ary = jnp.array(df_rho_div_f_ary)

theta = jnp.array(theta)

log_like_np_jax(pt_sum_compressed, theta, npt_compressed, data, f_ary, df_rho_div_f_ary)

DeviceArray([         nan,  -7.48560456,  -7.80127722,  12.37218433,
              -2.7888035 ,   8.51215257,   8.50653595,  -6.94955245,
             -19.70526901,  -6.45648684], dtype=float64)